# Add & Norm
After the Multi-Head Attention layer, the next step in a standard Transformer block is typically the "Add & Norm" component.
This component consists of two parts:

1. Add (Residual Connection):
    - This refers to adding the input of the Multi-Head Attention layer (the embeddings + positional encodings) to the output of the Multi-Head Attention layer.
    - If we denote the input to the Multi-Head Attention as X and the output as MultiHeadAttention(X), the "Add" step calculates X+MultiHeadAttention(X).
    - This is known as a residual connection. It helps with training very deep neural networks by providing a direct path for gradients to flow through the network during backpropagation, mitigating the vanishing gradient problem.

2. Norm (Layer Normalization):
    - After adding the residual connection, Layer Normalization is applied to the resulting sum.
    - Layer Normalization normalizes the values across the features for each individual token (layer-wise), helping to stabilize the training process and allowing for higher learning rates.

![add-and-norm](./resources/add-and-norm.png)

Let's first use the same code in the previous note to compute the final output `O` from 2-head attention

In [1]:
import torch
import math

d_model = 8             # Dimension of the embeddings
h = 2                   # 2-head attention
d_k = int(d_model / h)  # Dimension of the W_K and W_Q vectors, d_model / h
d_v = d_k               # Dimension of the W_V vectors, same as d_k

# Assume we have three embeddings, each has dimension d_model
E1 = torch.rand(d_model)
E2 = torch.rand(d_model)
E3 = torch.rand(d_model)
E = torch.stack([E1, E2, E3])

# First head
W_Q_1 = torch.rand([d_model, d_k])
W_K_1 = torch.rand([d_model, d_k])
W_V_1 = torch.rand([d_model, d_v])

# Second head
W_Q_2 = torch.rand([d_model, d_k])
W_K_2 = torch.rand([d_model, d_k])
W_V_2 = torch.rand([d_model, d_v])

# Calculate Q, K, V for the first head
Q_1 = E @ W_Q_1
K_1 = E @ W_K_1
V_1 = E @ W_V_1

# Calculate Q, K, V for the second head
Q_2 = E @ W_Q_2
K_2 = E @ W_K_2
V_2 = E @ W_V_2

# Compute the attention matrices for both heads
O_1 = torch.softmax(Q_1 @ K_1.T / math.sqrt(d_k), dim=1) @ V_1
O_2 = torch.softmax(Q_2 @ K_2.T / math.sqrt(d_k), dim=1) @ V_2
W_O = torch.rand([h * d_v, d_model])

# Concat O1 and O2 then multiply with W_O
O = torch.cat((O_1, O_2), dim=1)
O = O @ W_O
O

tensor([[7.5376, 5.3714, 6.8749, 5.2783, 8.4057, 9.1695, 6.4572, 9.2676],
        [7.4554, 5.3087, 6.8190, 5.2205, 8.3512, 9.1162, 6.4296, 9.1946],
        [7.5491, 5.3861, 6.8922, 5.2651, 8.4384, 9.2201, 6.5066, 9.2959]])

Then, we will add `O` with the original embedding `E`

In [3]:
O_add = E + O
O_add

tensor([[ 8.0767,  5.3746,  7.7834,  5.4983,  9.1930,  9.8454,  7.2636,  9.3102],
        [ 7.5725,  6.1879,  7.4546,  5.2340,  8.7860,  9.3213,  6.5481,  9.4872],
        [ 7.7602,  6.2049,  7.0121,  6.2146,  8.4436, 10.1703,  6.9111,  9.7810]])

Layer normalization is crucial in the Transformer architecture for the following reasons:

1. **Stabilizing Training**: By normalizing the inputs to each layer, layer normalization ensures that the distribution of activations remains consistent throughout training. This helps prevent issues like exploding or vanishing gradients.

2. **Improved Convergence**: Normalization allows the model to converge faster during training by maintaining a stable range of activations, enabling the use of higher learning rates.

3. **Feature-wise Normalization**: Unlike batch normalization, which normalizes across the batch dimension, layer normalization normalizes across the feature dimension for each individual input. This makes it particularly well-suited for sequence models like Transformers, where the input size can vary.

4. **Independence from Batch Size**: Since layer normalization operates on individual inputs rather than across a batch, it is more robust to changes in batch size and works effectively even with small batch sizes.

5. **Enhancing Generalization**: By reducing internal covariate shift, layer normalization helps the model generalize better to unseen data.

In [5]:
import torch
torch.layer_norm(O_add, (d_model, ))

tensor([[ 1.7954e-01, -1.5315e+00, -6.1941e-03, -1.4532e+00,  8.8644e-01,
          1.2996e+00, -3.3535e-01,  9.6071e-01],
        [-1.0086e-03, -9.6088e-01, -8.2755e-02, -1.6222e+00,  8.4028e-01,
          1.2113e+00, -7.1115e-01,  1.3264e+00],
        [-3.6298e-02, -1.1217e+00, -5.5841e-01, -1.1149e+00,  4.4062e-01,
          1.6457e+00, -6.2886e-01,  1.3739e+00]])